# End-to-End Machine Learning Pipeline: NASA Airfoil Self-Noise Prediction
**Author:** Tariq Zeyad Jawad

## 1. Project Overview
This project demonstrates an end-to-end data engineering and machine learning workflow using **Apache Spark (PySpark)**. The objective is to process raw aerodynamic and acoustic test data from the **NASA Airfoil Self-Noise dataset** (licensed under CC BY 4.0) and build a scalable machine learning pipeline to predict airfoil sound levels (in decibels) based on physical and aerodynamic features.

### Workflow Stages:
1. **ETL & Data Cleaning:** Ingest raw CSV data, remove duplicate records and missing values, standardize column names, and store the optimized dataset in **Parquet** format.
2. **Feature Engineering & ML Pipeline:** Assemble features using `VectorAssembler`, normalize feature scales via `StandardScaler`, and train a `LinearRegression` model.
3. **Model Evaluation:** Assess model performance on unseen test data using RMSE/MSE, MAE, and $R^2$ metrics.
4. **Model Persistence:** Save the trained pipeline model to disk and reload it to simulate production inference.

## 2. Environment Setup & Library Imports

In [ ]:
# Install required packages
!pip install pyspark==3.1.2 -q
!pip install findspark -q

In [ ]:
# Suppress unnecessary warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.pipeline import PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("NASA_Airfoil_Noise_Prediction_Pipeline") \
    .getOrCreate()

## 3. Data Extraction, Transformation, and Loading (ETL)

In [ ]:
# Download the raw dataset
!wget -q -O NASA_airfoil_noise_raw.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv

In [ ]:
# Load raw dataset into a Spark DataFrame
raw_data_path = "NASA_airfoil_noise_raw.csv"
df = spark.read.csv(raw_data_path, header=True, inferSchema=True)

print(f"Initial row count: {df.count()}")
df.show(5)

In [ ]:
# Data Cleaning: Drop duplicate records and rows with missing values
df = df.dropDuplicates()
print(f"Row count after removing duplicates: {df.count()}")

df = df.dropna()
print(f"Row count after removing null values: {df.count()}")

# Rename target column for clarity
df = df.withColumnRenamed("SoundLevel", "SoundLevelDecibels")

# Save cleaned dataset in Parquet format for optimized I/O performance
cleaned_parquet_path = "NASA_airfoil_noise_cleaned.parquet"
df.write.mode("overwrite").parquet(cleaned_parquet_path)

## 4. Building the Machine Learning Pipeline

In [ ]:
# Load the cleaned Parquet dataset
df_cleaned = spark.read.parquet(cleaned_parquet_path)

# Stage 1: Assemble input features into a single feature vector
feature_cols = [
    "Frequency", 
    "AngleOfAttack", 
    "ChordLength", 
    "FreeStreamVelocity", 
    "SuctionSideDisplacement"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Stage 2: Standardize features using StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Stage 3: Define the Linear Regression estimator
lr = LinearRegression(featuresCol="scaledFeatures", labelCol="SoundLevelDecibels")

# Construct the 3-stage ML Pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])

In [ ]:
# Split the dataset into training (70%) and testing (30%) sets
training_data, testing_data = df_cleaned.randomSplit([0.70, 0.30], seed=42)

# Train the pipeline model
pipeline_model = pipeline.fit(training_data)

## 5. Model Evaluation

In [ ]:
# Generate predictions on the test dataset
predictions = pipeline_model.transform(testing_data)

# Evaluate using MSE, MAE, and R-Squared
evaluator_mse = RegressionEvaluator(predictionCol="prediction", labelCol="SoundLevelDecibels", metricName="mse")
evaluator_mae = RegressionEvaluator(predictionCol="prediction", labelCol="SoundLevelDecibels", metricName="mae")
evaluator_r2 = RegressionEvaluator(predictionCol="prediction", labelCol="SoundLevelDecibels", metricName="r2")

mse = evaluator_mse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)
lr_model = pipeline_model.stages[-1]

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-Squared (R2): {r2:.2f}")
print(f"Model Intercept: {lr_model.intercept:.2f}")

## 6. Model Persistence & Production Inference

In [ ]:
# Save the trained pipeline model to disk
model_save_path = "Airfoils_ML_Model"
pipeline_model.write().overwrite().save(model_save_path)

# Reload the persisted model
loaded_pipeline_model = PipelineModel.load(model_save_path)

# Run inference on test data using the loaded model
loaded_predictions = loaded_pipeline_model.transform(testing_data)
loaded_predictions.select("SoundLevelDecibels", "prediction").show(5)

In [ ]:
# Inspect feature coefficients from the loaded model
loaded_lr_model = loaded_pipeline_model.stages[-1]
input_cols = loaded_pipeline_model.stages[0].getInputCols()

print("Feature Coefficients:")
for col_name, coef in zip(input_cols, loaded_lr_model.coefficients):
    print(f" - {col_name}: {coef:.4f}")

# Stop the Spark Session
spark.stop()